# A Data-Driven Analysis of Tourist Satisfaction in Sri Lankan Destinations Using Online Review Analytics

## 1. Import Libraries

In [11]:
# Core Libraries
import pandas as pd
import numpy as np
import os
import re
import json
import random
import string
import subprocess
import sys
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Optional, Tuple
from urllib import error, request

# Data Processing & NLP
import pycountry
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.sentiment import SentimentIntensityAnalyzer

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.feature_extraction.text import CountVectorizer
from gensim import corpora
from gensim.models import LdaModel
from wordcloud import WordCloud

# Transformers & Deep Learning
from transformers import pipeline
import torch
import warnings
warnings.filterwarnings('ignore')

# Display Settings
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

# NLTK Downloads
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('vader_lexicon', quiet=True)

# Reproducibility
random.seed(42)
np.random.seed(42)

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


## 2. Load & Explore Data

In [12]:
# Load raw data
df = pd.read_csv("Reviews.csv", encoding='latin1')

# Basic inspection
print("📊 Dataset Overview:")
print(f"  Shape: {df.shape}")
print(f"  Columns: {df.columns.tolist()}")
print(f"\n📋 Data Types:")
print(df.dtypes)
print(f"\n❌ Missing Values:")
print(df.isnull().sum())
print(f"\n⚠️  Duplicates: {df.duplicated().sum()}")
print(f"\n⭐ Rating Distribution:")
print(df["Rating"].value_counts().sort_index())
print(f"\n📍 Top 10 Cities:")
print(df["Located_City"].value_counts().head(10))

📊 Dataset Overview:
  Shape: (16156, 14)
  Columns: ['Location_Name', 'Located_City', 'Location', 'Location_Type', 'User_ID', 'User_Location', 'User_Locale', 'User_Contributions', 'Travel_Date', 'Published_Date', 'Rating', 'Helpful_Votes', 'Title', 'Text']

📋 Data Types:
Location_Name           str
Located_City            str
Location                str
Location_Type           str
User_ID                 str
User_Location           str
User_Locale             str
User_Contributions    int64
Travel_Date             str
Published_Date          str
Rating                int64
Helpful_Votes         int64
Title                   str
Text                    str
dtype: object

❌ Missing Values:
Location_Name         0
Located_City          0
Location              0
Location_Type         0
User_ID               0
User_Location         0
User_Locale           0
User_Contributions    0
Travel_Date           0
Published_Date        0
Rating                0
Helpful_Votes         0
Title          

## 3. Feature Engineering: Location Extraction

In [13]:
# Define location mapping databases
provinces = [
    "Western Province", "Central Province", "Southern Province",
    "Northern Province", "Eastern Province", "North Western Province",
    "North Central Province", "Uva Province", "Sabaragamuwa Province",
]

city_to_district = {
    "Arugam Bay": "Ampara", "Colombo": "Colombo", "Kandy": "Kandy",
    "Nuwara Eliya": "Nuwara Eliya", "Galle": "Galle", "Mirissa": "Matara",
    "Ella": "Badulla", "Negombo": "Gampaha", "Polonnaruwa": "Polonnaruwa",
    "Sigiriya": "Matale", "Trincomalee": "Trincomalee", "Jaffna": "Jaffna",
    "Batticaloa": "Batticaloa", "Anuradhapura": "Anuradhapura", "Matara": "Matara",
    "Kalutara": "Kalutara", "Bentota": "Galle", "Hikkaduwa": "Galle",
    "Mirigama": "Gampaha", "Habarana": "Anuradhapura",
}

manual_location_mappings = {
    "Udawalawe National Park": {"province": "Sabaragamuwa Province", "district": "Ratnapura"},
    "North Central Province": {"province": "North Central Province", "district": "Anuradhapura"},
}

province_pattern = re.compile(
    r"(" + "|".join([re.escape(p) for p in provinces]) + r")", flags=re.IGNORECASE
)

def extract_province(location_str):
    """Extract province from location string"""
    if not isinstance(location_str, str) or not location_str.strip():
        return None
    
    loc = location_str.strip()
    
    # Manual mapping
    for k, v in manual_location_mappings.items():
        if k.lower() == loc.lower():
            return v.get("province")
    
    # Pattern matching
    m = province_pattern.search(loc)
    if m:
        return m.group(1)
    
    # Numeric pattern
    m2 = re.search(r"([A-Za-z ]+Province)\b", loc)
    if m2:
        return m2.group(1).strip()
    
    return None

def infer_district(row):
    """Infer district from location and city"""
    city = row.get("Located_City")
    location = row.get("Location")
    
    # Manual mapping
    if isinstance(location, str):
        for k, v in manual_location_mappings.items():
            if k.lower() == location.strip().lower():
                return v.get("district")
    
    # City mapping
    if isinstance(city, str) and city in city_to_district:
        return city_to_district[city]
    
    if not isinstance(location, str):
        return None
    
    parts = [p.strip() for p in location.split(",") if p.strip()]
    
    # Check for explicit district
    for part in parts:
        if "District" in part:
            return part.replace("District", "").strip()
    
    # Infer from position
    if len(parts) >= 3:
        return parts[-2]
    elif len(parts) == 2:
        return parts[0]
    elif len(parts) == 1 and not any(parts[0].lower() == p.lower() for p in provinces):
        return parts[0]
    
    return None

# Apply location extraction
print("🔄 Extracting province and district...")
df["province"] = df["Location"].apply(extract_province)
df["district"] = df.apply(infer_district, axis=1)

print(f"✓ Province coverage: {df['province'].notna().sum()}/{len(df)} rows")
print(f"✓ District coverage: {df['district'].notna().sum()}/{len(df)} rows")
print(f"\n📍 Top Provinces:")
print(df["province"].value_counts().head(10))

🔄 Extracting province and district...
✓ Province coverage: 16156/16156 rows
✓ District coverage: 16156/16156 rows

📍 Top Provinces:
province
Central Province          5252
North Central Province    3171
Southern Province         2648
Western Province          1890
Eastern Province          1162
Uva Province              1040
Sabaragamuwa Province      518
Northern Province          475
Name: count, dtype: int64


## 4. User Location Parsing: Country & Region

In [14]:
# Install pycountry if needed
try:
    import pycountry
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pycountry"])
    import pycountry

@dataclass
class ParseResult:
    country: Optional[str]
    other: Optional[str]
    method: str
    confidence: float

# Regional data
US_STATES = {"alabama", "alaska", "arizona", "arkansas", "california", "colorado",
             "connecticut", "delaware", "florida", "georgia", "hawaii", "idaho",
             "illinois", "indiana", "iowa", "kansas", "kentucky", "louisiana",
             "maine", "maryland", "massachusetts", "michigan", "minnesota",
             "mississippi", "missouri", "montana", "nebraska", "nevada",
             "new hampshire", "new jersey", "new mexico", "new york",
             "north carolina", "north dakota", "ohio", "oklahoma", "oregon",
             "pennsylvania", "rhode island", "south carolina", "south dakota",
             "tennessee", "texas", "utah", "vermont", "virginia", "washington",
             "west virginia", "wisconsin", "wyoming", "district of columbia", "puerto rico"}

AU_STATES = {"new south wales", "queensland", "south australia", "tasmania",
             "victoria", "western australia", "australian capital territory", "northern territory"}

CA_PROVINCES = {"alberta", "british columbia", "manitoba", "new brunswick",
                "newfoundland and labrador", "nova scotia", "ontario",
                "prince edward island", "quebec", "saskatchewan",
                "northwest territories", "nunavut", "yukon"}

INDIA_STATES = {"andhra pradesh", "arunachal pradesh", "assam", "bihar",
                "chhattisgarh", "goa", "gujarat", "haryana", "himachal pradesh",
                "jharkhand", "karnataka", "kerala", "madhya pradesh",
                "maharashtra", "manipur", "meghalaya", "mizoram", "nagaland",
                "odisha", "punjab", "rajasthan", "sikkim", "tamil nadu",
                "telangana", "tripura", "uttar pradesh", "uttarakhand",
                "west bengal", "delhi", "jammu and kashmir", "ladakh"}

UK_REGIONS = {"england", "scotland", "wales", "northern ireland"}

REGION_TO_COUNTRY = {
    **{state: "United States" for state in US_STATES},
    **{state: "Australia" for state in AU_STATES},
    **{state: "Canada" for state in CA_PROVINCES},
    **{state: "India" for state in INDIA_STATES},
    **{region: "United Kingdom" for region in UK_REGIONS},
    "new england": "United States",
}

# Build country index
def build_country_index():
    alias_map = {}
    for country in pycountry.countries:
        alias_map[country.name.lower()] = country.name
    
    manual = {
        "usa": "United States", "u.s.a": "United States", "us": "United States",
        "u.s": "United States", "uk": "United Kingdom", "u.k": "United Kingdom",
        "uae": "United Arab Emirates", "u.a.e": "United Arab Emirates",
        "russia": "Russian Federation", "viet nam": "Vietnam",
    }
    alias_map.update(manual)
    
    preferred = {
        "Korea, Republic of": "South Korea",
        "Korea, Democratic People's Republic of": "North Korea",
        "Russian Federation": "Russia", "Viet Nam": "Vietnam",
    }
    return alias_map, preferred

COUNTRY_ALIAS_TO_NAME, COUNTRY_SHORT_TO_PREFERRED = build_country_index()

def parse_user_location(raw_value):
    """Parse user location to extract country and region"""
    if raw_value is None or (isinstance(raw_value, float) and pd.isna(raw_value)):
        return ParseResult(country=None, other=None, method="empty", confidence=0.0)
    
    text = str(raw_value).strip()
    if not text:
        return ParseResult(country=None, other=None, method="empty", confidence=0.0)
    
    parts = [p.strip() for p in text.split(",") if p.strip()]
    country = None
    
    # Search for country
    for part in reversed(parts):
        if part.lower() in COUNTRY_ALIAS_TO_NAME:
            country = COUNTRY_SHORT_TO_PREFERRED.get(
                COUNTRY_ALIAS_TO_NAME[part.lower()],
                COUNTRY_ALIAS_TO_NAME[part.lower()]
            )
            break
    
    # Search for region
    region = None
    for part in reversed(parts):
        if part.lower() in REGION_TO_COUNTRY:
            region = part.title()
            if not country:
                country = REGION_TO_COUNTRY[part.lower()]
            break
    
    if country:
        return ParseResult(
            country=country, other=region,
            method="rule_based", confidence=0.9 if region else 0.85
        )
    
    return ParseResult(country=None, other=None, method="unresolved", confidence=0.0)

# Apply location parsing
print("🔄 Parsing user locations...")
parsed = df["User_Location"].apply(parse_user_location)
df["user_country"] = parsed.apply(lambda x: x.country)
df["user_region"] = parsed.apply(lambda x: x.other)
df["location_parse_confidence"] = parsed.apply(lambda x: x.confidence)

print(f"✓ Country resolution: {df['user_country'].notna().sum()}/{len(df)} rows")
print(f"\n🌍 Top 15 Countries:")
print(df["user_country"].value_counts().head(15))

🔄 Parsing user locations...
✓ Country resolution: 14494/16156 rows

🌍 Top 15 Countries:
user_country
United Kingdom          4530
Australia               2000
India                   1625
United States            950
United Arab Emirates     432
Canada                   396
Singapore                340
Germany                  307
New Zealand              240
France                   233
China                    214
Malaysia                 202
Ireland                  142
Belgium                  142
Switzerland              140
Name: count, dtype: int64


## 5. Feature Engineering: Temporal Features

In [15]:
# Convert dates
for col in ['Travel_Date', 'Published_Date']:
    df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
    df[f'{col}_Month'] = df[col].dt.month
    df[f'{col}_Year'] = df[col].dt.year

# Add text features
df['review_length'] = df['Text'].apply(len)
df['word_count'] = df['Text'].apply(lambda x: len(str(x).split()))
df['title_length'] = df['Title'].apply(len)

print("✓ Temporal features extracted")
print(f"\n📅 Date Range:")
print(f"  Travel: {df['Travel_Date'].min()} to {df['Travel_Date'].max()}")
print(f"  Published: {df['Published_Date'].min()} to {df['Published_Date'].max()}")

✓ Temporal features extracted

📅 Date Range:
  Travel: 2010-09-01 00:00:00+00:00 to 2023-05-01 00:00:00+00:00
  Published: 2011-03-12 12:00:09+00:00 to 2023-05-20 05:15:38+00:00


## 6. Data Cleanup & Consolidation

In [16]:
# Drop raw location columns (redundant with extracted features)
df = df.drop(columns=['Location', 'User_Location'], errors='ignore')

print(f"✓ Cleaned up redundant columns")
print(f"\n📊 Dataset Shape: {df.shape}")
print(f"\n📋 Current Columns ({len(df.columns)}):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

✓ Cleaned up redundant columns

📊 Dataset Shape: (16156, 24)

📋 Current Columns (24):
   1. Location_Name
   2. Located_City
   3. Location_Type
   4. User_ID
   5. User_Locale
   6. User_Contributions
   7. Travel_Date
   8. Published_Date
   9. Rating
  10. Helpful_Votes
  11. Title
  12. Text
  13. province
  14. district
  15. user_country
  16. user_region
  17. location_parse_confidence
  18. Travel_Date_Month
  19. Travel_Date_Year
  20. Published_Date_Month
  21. Published_Date_Year
  22. review_length
  23. word_count
  24. title_length


## 7. Install & Load Sentiment Analysis Models

In [17]:
# Install required packages
required_packages = ["torch", "transformers>=4.30.0", "sentence-transformers>=2.2.2"]
print("Installing required packages...")

for package in required_packages:
    pkg_name = package.split('>')[0].split('<')[0].split('=')[0].strip()
    try:
        __import__(pkg_name.replace('-', '_'))
        print(f"  ✓ {pkg_name} already installed")
    except ImportError:
        print(f"  Installing {pkg_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"  ✓ {pkg_name} installed")

print("\n✓ All packages ready")

Installing required packages...
  ✓ torch already installed
  ✓ transformers already installed
  ✓ sentence-transformers already installed

✓ All packages ready


In [18]:
# Load sentiment analysis models
print("Loading sentiment analysis models...")
print(f"Device: {'CUDA (GPU)' if torch.cuda.is_available() else 'CPU'}\n")

cardiff_sentiment = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    device=0 if torch.cuda.is_available() else -1
)
print("  ✓ Cardiff RoBERTa")

siebert_sentiment = pipeline(
    "text-classification",
    model="siebert/sentiment-roberta-large-english",
    device=0 if torch.cuda.is_available() else -1
)
print("  ✓ SieBERT")

bertweet_sentiment = pipeline(
    "text-classification",
    model="finiteautomata/bertweet-base-sentiment-analysis",
    device=0 if torch.cuda.is_available() else -1
)
print("  ✓ BERTweet")

try:
    absa_sentiment = pipeline(
        "text-classification",
        model="yangheng/deberta-v3-large-absa-v1.1",
        device=0 if torch.cuda.is_available() else -1
    )
    print("  ✓ DeBERTa ABSA")
except Exception as e:
    print(f"  ⚠️  DeBERTa ABSA failed, using Cardiff as fallback")
    absa_sentiment = cardiff_sentiment

emotion_classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    device=0 if torch.cuda.is_available() else -1
)
print("  ✓ Emotion Classifier")

print("\n✓ All models loaded successfully!")

Loading sentiment analysis models...
Device: CPU



Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6193.11it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ Cardiff RoBERTa


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 7873.26it/s]
RobertaForSequenceClassification LOAD REPORT from: siebert/sentiment-roberta-large-english
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ SieBERT


...

  ✓ BERTweet


Loading weights: 100%|██████████| 394/394 [00:00<00:00, 1695.67it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: yangheng/deberta-v3-large-absa-v1.1
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ DeBERTa ABSA


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 12996.19it/s]
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ Emotion Classifier

✓ All models loaded successfully!


## 8. Sentiment Analysis Helper Functions

In [19]:
def get_sentiment_safe(text, model):
    """Safely apply sentiment model with error handling"""
    try:
        if pd.isna(text) or (isinstance(text, str) and not text.strip()):
            return None, 0.0
        
        text_clean = str(text).strip()[:512]  # Truncate to 512 chars
        result = model(text_clean, truncation=True)[0]
        return result['label'], round(result['score'], 4)
    except Exception:
        return None, 0.0

def get_emotion_safe(text, model):
    """Get primary emotion from text"""
    try:
        if pd.isna(text) or (isinstance(text, str) and not text.strip()):
            return None
        
        text_clean = str(text).strip()[:512]
        result = model(text_clean, truncation=True)[0]
        return result['label']
    except Exception:
        return None

print("✓ Helper functions defined")

✓ Helper functions defined


## 9. Apply Multi-Model Sentiment Analysis

In [ ]:
# Step 1: Apply Cardiff RoBERTa to Title
print("🔄 Analyzing Title sentiment (Cardiff RoBERTa)...")
df[['title_cardiff_sentiment', 'title_cardiff_score']] = df['Title'].apply(
    lambda x: pd.Series(get_sentiment_safe(x, cardiff_sentiment))
)
print(f"  ✓ {df['title_cardiff_sentiment'].notna().sum()} rows processed")

🔄 Analyzing Title sentiment (Cardiff RoBERTa)...


In [ ]:
# Step 2: Apply Cardiff RoBERTa to Text
print("🔄 Analyzing Text sentiment (Cardiff RoBERTa)...")
df[['text_cardiff_sentiment', 'text_cardiff_score']] = df['Text'].apply(
    lambda x: pd.Series(get_sentiment_safe(x, cardiff_sentiment))
)
print(f"  ✓ {df['text_cardiff_sentiment'].notna().sum()} rows processed")

In [ ]:
# Step 3: Combine Title + Text for comprehensive analysis
print("🔄 Creating combined Title+Text...")
df['combined_text'] = (df['Title'].fillna('') + ' ' + df['Text'].fillna('')).str.strip()

# Step 3a: Apply Cardiff RoBERTa to Combined
print("🔄 Analyzing Combined sentiment (Cardiff RoBERTa)...")
df[['combined_cardiff_sentiment', 'combined_cardiff_score']] = df['combined_text'].apply(
    lambda x: pd.Series(get_sentiment_safe(x, cardiff_sentiment))
)
print(f"  ✓ {df['combined_cardiff_sentiment'].notna().sum()} rows processed")

In [ ]:
# Step 4: Apply SieBERT to Combined
print("🔄 Analyzing Combined sentiment (SieBERT)...")
df[['combined_siebert_sentiment', 'combined_siebert_score']] = df['combined_text'].apply(
    lambda x: pd.Series(get_sentiment_safe(x, siebert_sentiment))
)
print(f"  ✓ {df['combined_siebert_sentiment'].notna().sum()} rows processed")

In [ ]:
# Step 5: Apply BERTweet to Combined
print("🔄 Analyzing Combined sentiment (BERTweet)...")
df[['combined_bertweet_sentiment', 'combined_bertweet_score']] = df['combined_text'].apply(
    lambda x: pd.Series(get_sentiment_safe(x, bertweet_sentiment))
)
print(f"  ✓ {df['combined_bertweet_sentiment'].notna().sum()} rows processed")

In [ ]:
# Step 6: Apply DeBERTa ABSA to Combined
print("🔄 Analyzing Combined sentiment (DeBERTa ABSA)...")
df[['absa_best_sentiment', 'absa_best_score']] = df['combined_text'].apply(
    lambda x: pd.Series(get_sentiment_safe(x, absa_sentiment))
)
print(f"  ✓ {df['absa_best_sentiment'].notna().sum()} rows processed")

In [ ]:
# Step 7: Apply Emotion Classifier to Text
print("🔄 Analyzing Emotion (Text)...")
df['emotion'] = df['Text'].apply(lambda x: get_emotion_safe(x, emotion_classifier))
print(f"  ✓ {df['emotion'].notna().sum()} rows processed")

## 10. Ensemble Sentiment Voting

In [ ]:
# Create ensemble consensus from 4 models
def calculate_ensemble_sentiment(row):
    """Weighted voting from Cardiff, SieBERT, BERTweet, ABSA"""
    votes = []
    scores = []
    
    models = [
        (row['combined_cardiff_sentiment'], row['combined_cardiff_score']),
        (row['combined_siebert_sentiment'], row['combined_siebert_score']),
        (row['combined_bertweet_sentiment'], row['combined_bertweet_score']),
        (row['absa_best_sentiment'], row['absa_best_score']),
    ]
    
    for label, score in models:
        if pd.isna(label):
            continue
        
        norm_label = str(label).upper()
        if norm_label in ['NEGATIVE', 'NEG']:
            votes.append(-1)
        elif norm_label in ['POSITIVE', 'POS']:
            votes.append(1)
        else:
            votes.append(0)
        scores.append(score)
    
    if not votes:
        return 'NEUTRAL', 0.0
    
    weighted_sum = sum(v * s for v, s in zip(votes, scores))
    avg_score = sum(scores) / len(scores)
    
    if weighted_sum > 0.1:
        label = 'POSITIVE'
    elif weighted_sum < -0.1:
        label = 'NEGATIVE'
    else:
        label = 'NEUTRAL'
    
    return label, round(avg_score, 4)

print("🔄 Creating Ensemble Sentiment...")
df[['ensemble_sentiment', 'ensemble_score']] = df.apply(
    lambda row: pd.Series(calculate_ensemble_sentiment(row)), axis=1
)

print(f"✓ Ensemble created\n")
print("📊 Sentiment Distribution:")
print(df['ensemble_sentiment'].value_counts())
print(f"\n📈 Score Statistics:")
print(df['ensemble_score'].describe())

## 11. Save Enriched Dataset

In [ ]:
# Clean up temporary columns
if 'combined_text' in df.columns:
    df = df.drop(columns=['combined_text'])

# Save final dataset
output_file = "processed_tourism_reviews_with_sentiment.csv"
df.to_csv(output_file, index=False)

print(f"✅ Dataset saved: {output_file}")
print(f"\n📊 Final Statistics:")
print(f"  Rows: {len(df):,}")
print(f"  Columns: {len(df.columns)}")
print(f"\n📋 Feature Categories:")
print(f"  Original: Title, Text, Rating, etc.")
print(f"  Location: province, district, user_country, user_region")
print(f"  Temporal: Travel_Date/Year/Month, Published_Date/Year/Month")
print(f"  Text Features: review_length, word_count, title_length")
print(f"  Sentiment: title_cardiff, text_cardiff, combined_cardiff, combined_siebert, combined_bertweet, absa_best")
print(f"  Emotion: emotion")
print(f"  Ensemble: ensemble_sentiment, ensemble_score")